# **Installing rapidfuzz library to get access to fuzz and process modules, which will be used in fuzzy similarity later**

In [ ]:
!pip install rapidfuzz

Importing all necessary imports

In [ ]:
import os
import pandas as pd
import numpy as np
from rapidfuzz import fuzz, process

In [ ]:
pur_path=r"/content/input1.xlsx"
twob_path=r"/content/input2.xlsx"

Handelling inputs such that input files can be in csv or xlsx format

In [ ]:
def handle_multitype_import(file_path):

  if file_path is None:
    return None
  filename,ext=os.path.splitext(file_path)
  if ext==".csv":
    df=pd.read_csv(file_path)
  elif ext==".xlsx":
    df=pd.read_excel(file_path)
  else:
    print("file type not supported")
  return df

pur=handle_multitype_import(pur_path)
twob=handle_multitype_import(twob_path)

Data Normalization

1. normalization of column names
2. extracting preferred columns from the dataset
3. normalizing values of columns
    * Normalization of supplier names
    * Normalization of gstin values
    * Normalization of invoice numbers
    * Normalization of invoice dates


In [ ]:
#Column name identification and changing them to preferrd cols

preferred_column_names=["Supplier Name","GSTIN",
                        "Invoice No","Invoice Date",
                        "HSN Code (optional)","Taxable Value",
                        "IGST","CGST","SGST","Total GST"]

def normalize_col_names(column_name):
  return column_name.lower().strip().replace('_',' ').replace('-', ' ').replace('.', '')
from rapidfuzz import process, fuzz

def extract_preferred_columns_from_df(df,preferred_columns,threshold=95):

    df_cols = list(df.columns)
    df_cols_norm = {col: normalize_col_names(col)for col in df_cols}

    preferred_cols_norm = {pref: normalize_col_names(pref)for pref in preferred_columns}

    used_source_columns = set()
    rename_map = {}

    for pref_col, pref_norm in preferred_cols_norm.items():

        candidates = {
            src_col: norm_col
            for src_col, norm_col in df_cols_norm.items()
            if src_col not in used_source_columns
        }

        if not candidates:
            rename_map[pref_col] = None
            continue

        best_norm, score, _ = process.extractOne(
            pref_norm,
            candidates.values(),
            scorer=fuzz.token_set_ratio
        )

        if score >= threshold:
            matched_src_col = next(
                src for src, norm in candidates.items()
                if norm == best_norm
            )

            rename_map[matched_src_col] = pref_col
            used_source_columns.add(matched_src_col)
        else:
            rename_map[pref_col] = None

    final_rename_map = {
        src: tgt
        for src, tgt in rename_map.items()
        if src in df.columns and tgt is not None
    }

    renamed_df = df.rename(columns=final_rename_map)

    for col in preferred_columns:
        if col not in renamed_df.columns:
            renamed_df[col] = None

    renamed_df = renamed_df[preferred_columns]

    return renamed_df, rename_map

pur,pur_rename_map=extract_preferred_columns_from_df(pur,preferred_column_names)
twob,twob_rename_map=extract_preferred_columns_from_df(twob,preferred_column_names)

In [ ]:
def normalize_supplier_name(val):
    if pd.isna(val):
        return None
    s = str(val).lower().strip()
    return s


def normalize_gstin(val):
    if pd.isna(val):
        return None
    return str(val).strip().upper()


def normalize_invoice_no(val):
    if pd.isna(val):
        return None
    return str(val).strip().lower()



def normalize_invoice_date(series):

    dt = pd.to_datetime(
        series,
        errors="coerce",
        dayfirst=True
    )

    return (
        dt.dt.year * 10000 +
        dt.dt.month * 100 +
        dt.dt.day
    )


def normalize_hsn_code(val):
    if pd.isna(val):
        return None
    return str(val).strip()

def normalize_numeric_col(val):
  if pd.isna(val):
    return None
  return val


def normalize_values(df):

    if "Supplier Name" in df.columns:
        df["Supplier Name"] = df["Supplier Name"].apply(normalize_supplier_name)
    if "GSTIN" in df.columns:
        df["GSTIN"] = df["GSTIN"].apply(normalize_gstin)
    if "Invoice No" in df.columns:
        df["Invoice No"] = df["Invoice No"].apply(normalize_invoice_no)
    if "Invoice Date" in df.columns:
        df["Invoice Date"] = normalize_invoice_date(df["Invoice Date"])
    if "HSN Code (optional)" in df.columns:
        df["HSN Code (optional)"] = df["HSN Code (optional)"].apply(normalize_hsn_code)
    if "Taxable Value" in df.columns:
        df["Taxable Value"] = df["Taxable Value"].apply(normalize_numeric_col)
    if "IGST" in df.columns:
        df["IGST"] = df["IGST"].apply(normalize_numeric_col)
    if "CGST" in df.columns:
        df["CGST"] = df["CGST"].apply(normalize_numeric_col)
    if "SGST" in df.columns:
        df["SGST"] = df["SGST"].apply(normalize_numeric_col)
    if "Total GST" in df.columns:
        df["Total GST"] = df["Total GST"].apply(normalize_numeric_col)
    return df


pur = normalize_values(pur)
twob = normalize_values(twob)

locking down column names ones and for all

In [ ]:
FINAL_COLS = [
    "Supplier Name", "GSTIN", "Invoice No", "Invoice Date",
    "HSN Code (optional)", "Taxable Value",
    "IGST", "CGST", "SGST", "Total GST"
]

pur = pur[FINAL_COLS]
twob = twob[FINAL_COLS]

1. similarity_threshold governs fuzzy similarity cutoff i.e "match" if i>T else "not match"
2. numeric_tolerance governs tolerance in numeric values in matching i.e 1 percent tolerance means a value can be +- 1 percent to match
3. we cant use fuzzy match for all the values in df2 for each value of df1, complexity will go through the roof
4. to counter that  we will be selecting candidates first, these candidates are selected based on exactly matching GSTIN and invoice date and tolerance matching Taxable value
5. Once we have the candidates, process will pick one best depending upon WRatio match between candidates
6. As a fallback measure, if we fail to determine match above threshold then all the candidates will be saved as probable matches in differnt column

In [ ]:
def reconcile(pur, twob, similarity_threshold=80, numeric_tolerance=0.01):

    #result columns
    pur["Best score"] = pd.NA
    pur["Best match 2B index"] = pd.NA
    pur["Probable 2B indexes"] = pd.NA

    twob["Best score"] = pd.NA
    twob["Best match PR index"] = pd.NA
    twob["Probable PR indexes"] = pd.NA

    matched_2b_indexes = set()

    for row in pur.itertuples():

        #hard filters
        candidates = twob[
            (twob["GSTIN"] == row.GSTIN) &
            (twob["Invoice Date"] == row._4) &
            (abs(twob["Taxable Value"] - row._6) <= numeric_tolerance * max(row._6, 1))
            #&(abs(twob["IGST"] - row.IGST) <= numeric_tolerance * max(row.IGST, 1)) &
            #(abs(twob["CGST"] - row.CGST) <= numeric_tolerance * max(row.CGST, 1)) &
            #(abs(twob["SGST"] - row.SGST) <= numeric_tolerance * max(row.SGST, 1)) &
            #(abs(twob["Total GST"] - row._10) <= numeric_tolerance * max(row._10, 1))
        ]

        if candidates.empty:
            continue

        best_score = 0
        best_candidate_index = None
        candidate_indexes = []

        for candidate in candidates.itertuples():

            candidate_indexes.append(candidate.Index)

            if candidate.Index in matched_2b_indexes:
                continue

            score = (
                0.5 * fuzz.WRatio(row._1, candidate._1) +
                0.5 * fuzz.WRatio(str(row._3), str(candidate._3))
            )

            if score > best_score:
                best_score = score
                best_candidate_index = candidate.Index

        # strong match
        if best_candidate_index is not None and best_score >= similarity_threshold:

            pur.at[row.Index, "Best score"] = best_score
            pur.at[row.Index, "Best match 2B index"] = best_candidate_index

            twob.at[best_candidate_index, "Best score"] = best_score
            twob.at[best_candidate_index, "Best match PR index"] = row.Index

            matched_2b_indexes.add(best_candidate_index)

        #probablre batch
        else:
            pur.at[row.Index, "Probable 2B indexes"] = candidate_indexes

            for idx in candidate_indexes:
                existing = twob.at[idx, "Probable PR indexes"]
                if pd.isna(existing):
                    twob.at[idx, "Probable PR indexes"] = [row.Index]
                else:
                    twob.at[idx, "Probable PR indexes"].append(row.Index)

    return pur, twob

pur,twob=reconcile(pur,twob)

In [ ]:
pur.sample(10)

,Supplier Name,GSTIN,Invoice No,Invoice Date,HSN Code (optional),Taxable Value,IGST,CGST,SGST,Total GST,Best score,Best match 2B index,Probable 2B indexes
481,manilal sanghvi (bombay),27AAAFM2359N1ZZ,7930,20240621,None,44419.5,0.0,3997.76,3997.76,7995.52,98.93617,440,<NA>
183,makharia machineries pvt. ltd.,27AAACM2855Q1ZY,mh/1368/24-25,20240506,None,18547.2,0.0,1669.25,1669.25,3338.50,92.424242,282,<NA>
243,r.k.engineering corporation,27AAEFR8996A1ZU,699/24-25,20240514,None,6490.8,0.0,584.17,584.17,1168.34,87.5,311,<NA>
100,bharat bijlee limited,27AAACB2900K1ZZ,4320291106,20240422,None,49550.0,0.0,4460.00,4460.00,8920.00,100.0,76,<NA>
35,punaghar mata tool centre,27CBSPS4453E1Z6,2024-25/apr031,20240406,None,150.0,0.0,13.50,13.50,27.00,99.019608,120,<NA>
352,bharat bijlee limited,27AAACB2900K1ZZ,4320294865,20240601,None,72012.0,0.0,6481.00,6481.00,12962.00,100.0,384,<NA>
295,bharat bijlee limited,27AAACB2900K1ZZ,4320293860,20240522,None,70242.0,0.0,6322.00,6322.00,12644.00,100.0,225,<NA>
462,asiatic gases limited,27AACCA4768Q1Z0,567,20240618,None,565.5,0.0,50.90,50.90,101.80,100.0,533,<NA>
212,mould n cast,27AAOPS6151N1ZW,61,20240511,None,775.0,0.0,69.75,69.75,139.50,100.0,293,<NA>
53,manilal sanghvi (bombay),27AAAFM2359N1ZZ,7719,20240410,None,31027.5,0.0,2792.48,2792.48,5584.96,98.93617,11,<NA>


In [ ]:
twob.sample(10)

,Supplier Name,GSTIN,Invoice No,Invoice Date,HSN Code (optional),Taxable Value,IGST,CGST,SGST,Total GST,Best score,Best match PR index,Probable PR indexes
229,bharat bijlee limited,27AAACB2900K1ZZ,4320294176,20240525,None,33118.0,0.0,2981.00,2981.00,5962.00,100.0,304,<NA>
144,icici bank limited,27AAACI1195H1ZM,0135240411052762,20240411,None,1000.0,0.0,90.00,90.00,180.00,<NA>,<NA>,<NA>
392,bharat bijlee limited,27AAACB2900K1ZZ,4320295224,20240606,None,17364.0,0.0,1563.00,1563.00,3126.00,100.0,389,<NA>
453,emco engineering,27AACFE6426A1ZW,250774,20240627,None,7922.2,0.0,713.00,713.00,1426.00,100.0,533,<NA>
62,bharat bijlee limited,27AAACB2900K1ZZ,4320289699,20240403,None,119032.0,0.0,10713.00,10713.00,21426.00,100.0,16,<NA>
67,bharat bijlee limited,27AAACB2900K1ZZ,4320289899,20240405,None,136806.0,0.0,12313.00,12313.00,24626.00,100.0,28,<NA>
57,shah & company jaliwala,27AAMFS8359L1Z9,5672,20240401,None,21000.0,0.0,1890.00,1890.00,3780.00,<NA>,<NA>,[0]
227,bharat bijlee limited,27AAACB2900K1ZZ,4320293866,20240522,None,55212.0,0.0,4969.00,4969.00,9938.00,100.0,297,<NA>
297,kalapurnam industrial traders,27ACIPK1680N1Z6,355,20240521,None,1050.0,0.0,94.50,94.50,189.00,94.152542,283,<NA>
542,maharashtra pipe centre,27ESDPK7454P1ZY,107,20240619,None,593.4,0.0,53.41,53.41,106.82,100.0,469,<NA>


In [ ]:
#If you want to export


pur.to_excel("pr_reconciled.xlsx")
twob.to_excel("gstr2b_reconciled.xlsx")
